# EDA approfondie — Home Credit

Cette analyse travaille sur la table `application_train.csv` complète. Elle décrit la qualité des données, les distributions, les valeurs extrêmes, les associations avec le défaut et trois hypothèses pré-définies. Les données brutes ne sont jamais modifiées.

**Précaution :** les associations et tests statistiques ne démontrent pas de causalité. Les grands effectifs rendent les tests sensibles à de faibles écarts ; on rapporte également les médianes et une taille d'effet. Aucun seuil d'outlier ne justifie à lui seul la suppression d'une observation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import mannwhitneyu

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data" / "raw").exists():
    ROOT_DIR = ROOT_DIR.parent
sys.path.insert(0, str(ROOT_DIR))

RAW_DIR = ROOT_DIR / "data" / "raw"
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
print(f"Racine du projet : {ROOT_DIR}")

## 1. Chargement et dimensions

La table d'entraînement contient la cible `TARGET`. La table de test de la compétition n'en contient pas et ne doit pas servir à calculer des statistiques de cible.

In [ ]:
train_path = RAW_DIR / "home_credit" / "application_train.csv"
if not train_path.is_file():
    raise FileNotFoundError(f"Jeu Home Credit introuvable : {train_path}")

applications = pd.read_csv(train_path, low_memory=False)
print(f"Dimensions : {applications.shape[0]:,} lignes × {applications.shape[1]} colonnes")
print(f"Mémoire pandas : {applications.memory_usage(deep=True).sum() / 1024**2:,.1f} MiB")
applications.info(memory_usage="deep")

## 2. Qualité et valeurs manquantes

On contrôle les identifiants, les doublons exacts, la cible et les valeurs manquantes. La proportion manquante est calculée colonne par colonne ; une forte proportion appelle une décision documentée, pas une suppression automatique.

In [ ]:
quality_report = pd.DataFrame(
    {
        "type": applications.dtypes.astype(str),
        "valeurs_distinctes": applications.nunique(dropna=False),
        "valeurs_manquantes": applications.isna().sum(),
        "manquants_pct": applications.isna().mean().mul(100),
    }
).sort_values("manquants_pct", ascending=False)

print(f"Identifiants client dupliqués : {applications['SK_ID_CURR'].duplicated().sum():,}")
print(f"Lignes entièrement dupliquées : {applications.duplicated().sum():,}")
print(f"Cibles manquantes : {applications['TARGET'].isna().sum():,}")
print(f"Colonnes sans valeur manquante : {applications.isna().sum().eq(0).sum()} / {applications.shape[1]}")
display(quality_report.head(20).round(2))

In [ ]:
missing_top = quality_report.loc[quality_report["valeurs_manquantes"].gt(0)].head(20)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(
    data=missing_top.reset_index(names="variable"),
    x="manquants_pct",
    y="variable",
    color="steelblue",
    ax=axes[0],
)
axes[0].set(title="20 variables les plus incomplètes", xlabel="Valeurs manquantes (%)", ylabel="")

target_counts = applications["TARGET"].value_counts().sort_index()
sns.barplot(x=target_counts.index.astype(str), y=target_counts.values, color="darkorange", ax=axes[1])
axes[1].set(title="Déséquilibre de la cible", xlabel="TARGET (1 = défaut)", ylabel="Nombre de dossiers")
plt.tight_layout()
plt.show()

missing_by_target = applications.groupby("TARGET")[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].agg(
    lambda values: values.isna().mean() * 100
)
display(missing_by_target.rename(index={0: "Sans défaut", 1: "Défaut"}).round(2))

## 3. Distributions et valeurs extrêmes

Les bornes de Tukey (`Q1 − 1,5 × IQR`, `Q3 + 1,5 × IQR`) servent ici uniquement à quantifier des valeurs à examiner. Elles ne sont pas un filtre de nettoyage. `DAYS_EMPLOYED=365243` est inspecté séparément : sa valeur dépasse la durée plausible d'un emploi et doit être traitée comme code/anomalie, pas comme une ancienneté valide.

In [ ]:
amount_columns = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
outlier_rows = []
for column in amount_columns:
    values = applications[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outside = values.lt(lower_bound) | values.gt(upper_bound)
    outlier_rows.append(
        {
            "variable": column,
            "borne_basse": lower_bound,
            "borne_haute": upper_bound,
            "hors_bornes": int(outside.sum()),
            "hors_bornes_pct": outside.mean() * 100,
            "maximum": values.max(),
        }
    )

outlier_summary = pd.DataFrame(outlier_rows).set_index("variable")
display(outlier_summary.round(2))
print(
    "DAYS_EMPLOYED=365243 : "
    f"{applications['DAYS_EMPLOYED'].eq(365243).sum():,} lignes "
    f"({applications['DAYS_EMPLOYED'].eq(365243).mean() * 100:.2f} %)."
)

In [ ]:
plot_data = applications[["TARGET", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH", "AMT_CREDIT"]].copy()
plot_data["AGE_YEARS"] = -plot_data["DAYS_BIRTH"] / 365.25

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(data=plot_data, x="EXT_SOURCE_2", hue="TARGET", stat="density", common_norm=False, bins=40, ax=axes[0, 0])
axes[0, 0].set_title("Score externe 2 selon le défaut")
sns.histplot(data=plot_data, x="EXT_SOURCE_3", hue="TARGET", stat="density", common_norm=False, bins=40, ax=axes[0, 1])
axes[0, 1].set_title("Score externe 3 selon le défaut")
sns.histplot(data=plot_data, x="AGE_YEARS", hue="TARGET", stat="density", common_norm=False, bins=40, ax=axes[1, 0])
axes[1, 0].set_title("Âge au moment de la demande")
sns.histplot(data=plot_data, x="AMT_CREDIT", hue="TARGET", stat="density", common_norm=False, bins=40, log_scale=True, ax=axes[1, 1])
axes[1, 1].set_title("Montant du crédit (échelle logarithmique)")
plt.tight_layout()
plt.show()

## 4. Associations avec la cible

Les corrélations de Pearson avec `TARGET` sont des résumés linéaires, pas des mesures de causalité. Les déciles du score externe 2 rendent la relation plus lisible, tout en rappelant que le taux observé dépend de cette population de demandes.

In [ ]:
target_correlations = (
    applications.select_dtypes(include="number")
    .corr()["TARGET"]
    .drop(labels=["TARGET", "SK_ID_CURR"], errors="ignore")
    .dropna()
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .head(12)
)
display(target_correlations.rename("corrélation de Pearson avec TARGET").to_frame().round(4))

fig, ax = plt.subplots(figsize=(8, 6))
target_correlations.sort_values().plot.barh(ax=ax, color="slateblue")
ax.set(title="Variables numériques les plus associées à TARGET", xlabel="Corrélation de Pearson", ylabel="")
plt.tight_layout()
plt.show()

score_deciles = pd.qcut(applications["EXT_SOURCE_2"], q=10, duplicates="drop")
risk_by_score = applications.groupby(score_deciles, observed=True)["TARGET"].agg(
    dossiers="size", taux_defaut="mean"
)
risk_by_score["taux_defaut_pct"] = risk_by_score["taux_defaut"] * 100
display(risk_by_score.drop(columns="taux_defaut").round(2))

## 5. Hypothèses statistiques pré-définies

Pour chaque variable, l'hypothèse nulle est que sa distribution est identique chez les dossiers en défaut et sans défaut. Le test de Mann–Whitney bilatéral compare les distributions ; l'effet rang-bisérial est positif si les valeurs tendent à être plus élevées chez les dossiers en défaut. Une correction de Bonferroni couvre les trois tests.

1. `EXT_SOURCE_2` est associé au statut de défaut.
2. `EXT_SOURCE_3` est associé au statut de défaut.
3. L'âge (dérivé de `DAYS_BIRTH`) est associé au statut de défaut.

La très grande taille de l'échantillon peut rendre statistiquement significatifs des écarts modestes : considérer ensemble la taille d'effet, les médianes, le contexte métier et les facteurs confondants.

In [ ]:
hypotheses = [
    ("EXT_SOURCE_2", "Score externe 2"),
    ("EXT_SOURCE_3", "Score externe 3"),
    ("DAYS_BIRTH", "Âge (jours avant la demande)"),
]
test_results = []
for column, label in hypotheses:
    default_values = applications.loc[applications["TARGET"].eq(1), column].dropna()
    non_default_values = applications.loc[applications["TARGET"].eq(0), column].dropna()
    result = mannwhitneyu(default_values, non_default_values, alternative="two-sided", method="asymptotic")
    p_value = float(result.pvalue)
    adjusted_p_value = min(p_value * len(hypotheses), 1.0)
    rank_biserial = 2 * result.statistic / (len(default_values) * len(non_default_values)) - 1

    test_results.append(
        {
            "variable": label,
            "n_défaut": len(default_values),
            "n_sans_défaut": len(non_default_values),
            "médiane_défaut": default_values.median(),
            "médiane_sans_défaut": non_default_values.median(),
            "effet_rang_bisérial": rank_biserial,
            "p_brute": p_value,
            "p_bonferroni": adjusted_p_value,
        }
    )

hypothesis_results = pd.DataFrame(test_results).set_index("variable")
display(hypothesis_results.round(6))
print(
    "Si une p-value s'affiche à 0, c'est un sous-flux numérique de SciPy, "
    "pas une probabilité exactement nulle. Dans ce cas, interpréter le résultat comme p < 1e-300."
)

## 6. Taux observés par groupes

Ces tableaux servent à décrire la population et à ouvrir une revue d'équité ; les variables sensibles ne sont pas automatiquement recommandées comme variables de décision. Les groupes de très petite taille (par exemple `XNA`) ne permettent pas une conclusion fiable.

In [ ]:
for column in ["CODE_GENDER", "NAME_CONTRACT_TYPE"]:
    group_rates = applications.groupby(column, dropna=False)["TARGET"].agg(
        dossiers="size", taux_defaut="mean"
    )
    group_rates["taux_defaut_pct"] = group_rates["taux_defaut"] * 100
    print(f"\nTaux de défaut par {column} :")
    display(group_rates.drop(columns="taux_defaut").round(2))

## 7. Nettoyage prudent et décisions

Le pré-nettoyage transforme uniquement le code impossible `DAYS_EMPLOYED=365243` en valeur manquante et ajoute un indicateur. Les lignes et les identifiants restent inchangés. Les autres valeurs manquantes seront imputées dans un pipeline ajusté sur l'entraînement ; aucun outlier n'est tronqué à cette étape.

Cette stratégie distingue l'exploration du prétraitement appris, évite de modifier les fichiers sources et prépare la validation anti-fuite de l'étape suivante.

In [ ]:
from src.data.home_credit import clean_application_data

cleaning_example = applications[["SK_ID_CURR", "DAYS_EMPLOYED"]].copy()
cleaning_result = clean_application_data(cleaning_example)
print(f"Lignes conservées : {len(cleaning_result):,} / {len(cleaning_example):,}")
print(
    "Codes d'emploi transformés en manquants : "
    f"{cleaning_result['DAYS_EMPLOYED_ANOMALY'].sum():,}"
)
display(cleaning_result["DAYS_EMPLOYED_ANOMALY"].value_counts().rename("effectif"))
print(f"Valeurs manquantes de DAYS_EMPLOYED après nettoyage : {cleaning_result['DAYS_EMPLOYED'].isna().sum():,}")

## Synthèse à retenir

- `TARGET` est déséquilibrée : privilégier des métriques telles que PR-AUC et rappel en plus de ROC-AUC.
- Les scores `EXT_SOURCE_1/2/3` sont les associations linéaires les plus marquées avec `TARGET` dans cette table ; vérifier leur disponibilité et leur définition avant la modélisation.
- Les caractéristiques du bâtiment et du logement ont de fortes proportions de valeurs manquantes ; conserver l'information de manque et comparer des choix d'imputation dans des pipelines.
- Examiner les valeurs extrêmes avec le dictionnaire métier. Les bornes statistiques ne justifient pas de supprimer les dossiers.
- Garder un découpage train/validation avant l'apprentissage des encodeurs et imputeurs ; ne pas utiliser `application_test.csv` pour guider les transformations liées à la cible.